# 07 - Spend Caps

Without spend caps, a single bug or runaway loop can drain a user's entire credit balance in seconds. Spend caps act as safety valves — they limit how many credits a user can consume in a given period, protecting both the user and the platform operator from unexpected costs.

bursar supports three cap behaviors: **deny** (hard block — the operation is rejected when the cap is exceeded), **warn** (soft alert — the operation proceeds but the overage is flagged), and **notify** (passive monitoring — the operation proceeds and the overage is recorded the same way). The deny action is the most common choice for production deployments, while warn and notify are useful for gradual rollouts or monitoring-only scenarios.

Caps can be configured per user and per period type. bursar supports **daily** caps, which reset every calendar day, and **monthly** caps, which reset every calendar month. You can set different limits for different users — for example, a 5,000-credit daily cap for free-tier users and a 50,000-credit daily cap for enterprise customers.

In this notebook we will use the `MemoryStore` implementation and walk through each cap action type. You will see what happens when a deduction stays under the cap, when it exceeds a deny cap, when it exceeds a warn cap, and when it exceeds a notify cap. By the end of this notebook you will understand how to configure and enforce spend controls in your own bursar deployment.

### Setup

This notebook uses `MemoryStore` because writing a spend cap (`set_spend_cap`) is a `MemoryStore`-only convenience, not part of the portable `CreditStore` interface (see Notebook 00). `PostgresStore` ships the same `credit_spend_caps` schema and implements the read side (`check_spend_cap`, which `deduct_with_allowance` also calls internally) — but there is no writer method. In production you configure a cap by inserting a row into `credit_spend_caps` directly (via your own admin tooling or a migration); Postgres then enforces it exactly like `MemoryStore` does below.

The `memory_setup()` helper creates a fresh `MemoryStore` instance, imports the `SpendCap` model (which defines cap configurations), and initializes the store's internal tables. All of our spend cap examples will run against this in-memory store.

In [ ]:
import uuid
from datetime import datetime, timedelta
from bursar.interface.memory import MemoryStore
from bursar.manager import CreditManager
from bursar.engine import PricingEngine
from bursar.metrics import UsageMetrics, ToolCall
from bursar.interface.models import (
    PricingConfigData, PlanDefinition,
    CreditMetadata, SpendCap,
)

store = MemoryStore()
store.setup()
print("✔ MemoryStore ready.")

### Set a daily deny-cap of 5 000

A deny cap is the strictest form of spend control: if the user's cumulative spend for the period reaches the limit, any further deduction attempt is rejected with an error. No credits leave the account, and the calling application must handle the rejection gracefully.

In this section we create a user, seed their account with 50,000 credits, and then configure a daily deny cap of 5,000 credits. The `SpendCap` model takes four parameters: the user identifier, the cap type (`"daily"` or `"monthly"`), the credit limit (in whole credits), and the action to take when the limit is reached (`"deny"`, `"warn"`, or `"notify"`). This sets up the conditions for the next two sections, where we test the cap from both sides: under and over.

In [ ]:
# Create a new user for this spend cap demonstration
user = str(uuid.uuid4())

# Seed the user with a generous starting balance of 50,000 credits
store.add_credits(user, 50_000, type="seed")
print(f"  Initial balance: {store.get_balance(user).balance}")

# Define a daily deny-cap: maximum 5,000 credits consumed per day, hard block
cap = SpendCap(user_id=user, cap_type="daily", limit=5_000, action="deny")

# Register the cap with the store
store.set_spend_cap(cap)
print(f"  Cap registered: type=daily, limit={cap.limit}, action={cap.action}")

### Deduct under cap (succeeds)

With the 5,000-credit daily deny cap in place, we first test a deduction that stays under the limit. A 3,000-credit charge is well within the 5,000-credit cap, so the deduction should complete successfully.

After the deduction, we call `check_spend_cap()` to inspect the user's current spend tracking. This method returns a `CapCheckResult` object that tells us whether the user is currently capped, how much they have spent in the current period, and what their limit is. After a 3,000-credit deduction against a 5,000-credit cap, the current spend should be 3,000 and the user should not be flagged as capped.

In [ ]:
# Deduct 3,000 credits — this is well under the 5,000 daily cap.
# deduct_with_allowance() enforces the cap as part of its atomic transaction,
# so this single call both charges the user and checks the cap.
ded = store.deduct_with_allowance(user, 3_000)
print(f"  Deduction succeeded: balance after = {ded.balance_after}")

# Check the user's current spend against their configured cap
check = store.check_spend_cap(user)
print(f"  Cap status: capped={check.capped}, current spend={check.current_spend}")

### Exceed cap (denied)

Now we attempt a second deduction that would push the user over the 5,000-credit daily limit. The user has already spent 3,000 credits, and a further 3,000-credit deduction would bring the total to 6,000 — exceeding the cap by 1,000.

With the deny action, the store rejects the deduction and returns an error message explaining why. The credits stay in the user's account: the cap check and the balance debit happen inside the same server-side transaction as `deduct_with_allowance` itself, so there is no window where a concurrent request could sneak a charge through between checking the cap and applying it. This is the expected behavior for a hard cap: the application receives the error and can decide how to respond — perhaps by showing the user an upgrade prompt, logging the event for admin review, or retrying with a smaller operation.

This safety net prevents runaway costs from a single misconfigured loop or a compromised API key. Without it, the same 3,000-credit deduction would succeed and leave the platform operator to detect the overage retroactively. In this section we attempt the over-cap deduction and observe the deny behavior in action.

In [ ]:
# Attempt a second deduction of 3,000 credits — this would exceed the remaining
# cap (3,000 already spent + 3,000 new > 5,000 cap). deduct_with_allowance()
# rejects it atomically, before any credits leave the account.
ded2 = store.deduct_with_allowance(user, 3_000)

# Check the result: with a deny cap, the error field contains a descriptive message
if ded2.error:
    print(f"  Deduction denied: {ded2.error}")
else:
    print(f"  Deduction allowed: balance after = {ded2.balance_after}")
print(f"  Explanation: daily cap = 5,000, already spent 3,000 today")

### Cap with warn action

Not every cap breach needs to be a hard block. Sometimes you want to allow the operation but flag it for attention. The **warn** action does exactly that: the deduction proceeds normally (`result.error` stays `None`), but `check_spend_cap()` reports that the user is over their configured limit.

This is useful for gradual rollouts of spend controls. You might set a warn cap first to observe how many users would be affected, then switch to deny once you are confident in the limits. It is also appropriate for internal tools or trusted users where you want visibility into spending without disrupting their workflow.

In this section we create a second user with a much lower daily cap of 500 credits and the warn action. We then attempt a 1,000-credit deduction and observe that it succeeds even though it exceeds the cap. The `check_spend_cap()` response confirms the overage but does not block the operation.

In [ ]:
# Create a second user for the warn cap demonstration
user2 = str(uuid.uuid4())

# Seed the user with 50,000 credits, same as the first user
store.add_credits(user2, 50_000, type="seed")

# Set a very low daily cap of 500 credits with the warn action (not deny)
store.set_spend_cap(SpendCap(user_id=user2, cap_type="daily", limit=500, action="warn"))

# Attempt to deduct 1,000 credits — exceeds the 500-credit warn cap, but a
# warn cap never blocks the deduction, only flags it.
ded3 = store.deduct_with_allowance(user2, 1_000)
print(f"  Deduction result: error={ded3.error}  (None -- warn never blocks)")

# Check cap status — the action field confirms "warn" and current spend exceeds the limit
check2 = store.check_spend_cap(user2)
print(f"  Current spend: {check2.current_spend}  Cap limit: {check2.cap_limit}  Action: {check2.action}")
print(f"  Key insight: warn action allows the deduction but flags the overage")

### Cap with notify action

The **notify** action behaves identically to **warn** at the deduction level — the charge still proceeds, and `check_spend_cap()` still reports the overage. bursar doesn't treat the two differently in the store; the distinction is a labeling convention for *your* application to interpret. A common split: surface `warn` inline (e.g. a banner in the product UI), and route `notify` to an out-of-band channel your users don't see directly (e.g. an email digest or a Slack alert to the account team) — both read the same `action` field from `check_spend_cap()` to decide which path to take.

In [ ]:
# Create a third user for the notify cap demonstration
user3 = str(uuid.uuid4())
store.add_credits(user3, 50_000, type="seed")
store.set_spend_cap(SpendCap(user_id=user3, cap_type="daily", limit=1_000, action="notify"))

# Exceeds the 1,000-credit notify cap — proceeds anyway, just like warn.
ded4 = store.deduct_with_allowance(user3, 1_500)
check3 = store.check_spend_cap(user3)
print(f"  Deduction result: error={ded4.error}  balance_after={ded4.balance_after}")
print(f"  Cap status: current spend={check3.current_spend}  limit={check3.cap_limit}  action={check3.action}")
print("  Key insight: notify behaves identically to warn at the deduction level --")
print("  it's up to your application to route action=='notify' differently from action=='warn'.")